# 🎭 Emotion Detection & Sarcasm Analysis

**Pipeline:**
1. **Emotion Model** — Fine-tuned `distilroberta-base` on GoEmotions → Ekman 6 emotions with confidence + word attribution
2. **Sarcasm Model** — `mrm8488/t5-base-finetuned-sarcasm-twitter` (zero-shot, no fine-tuning needed)
3. **Classical Baselines** — TF-IDF + {Logistic Regression, SVM, Random Forest, Naive Bayes}
4. **Streamlit Dashboard** — Interactive text input with confidence bars and token attribution heatmap

**Ekman 6 emotions:** `joy`, `sadness`, `anger`, `fear`, `disgust`, `surprise`

## 0. Install Dependencies

In [ ]:
%%capture
!pip install transformers datasets torch scikit-learn pandas numpy matplotlib seaborn \
             shap captum streamlit plotly nltk accelerate evaluate \
             sentencepiece protobuf --quiet

## 1. Imports & Configuration

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline,
)
import evaluate

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 80)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ─── Global Config ───────────────────────────────────────────────────────────
EMOTION_MODEL_CKPT  = "distilbert/distilroberta-base"
SARCASM_MODEL_CKPT  = "mrm8488/t5-base-finetuned-sarcasm-twitter"
DATA_DIR            = Path("GoEmotions/data")          # GoEmotions TSVs + mapping files
OUTPUT_DIR          = Path("./checkpoints/emotion_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN       = 128
BATCH_SIZE    = 32
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01

# Ekman 6 (+ neutral). The fine→coarse mapping is loaded from the dataset's
# ekman_mapping.json in section 2 rather than hardcoded here.
EKMAN_LABELS   = ['joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise', 'neutral']
NUM_LABELS     = len(EKMAN_LABELS)
LABEL2ID       = {l: i for i, l in enumerate(EKMAN_LABELS)}
ID2LABEL       = {i: l for l, i in LABEL2ID.items()}

EMOTION_COLORS = {
    'joy':      '#FFD700',
    'sadness':  '#4169E1',
    'anger':    '#DC143C',
    'fear':     '#800080',
    'disgust':  '#228B22',
    'surprise': '#FF8C00',
    'neutral':  '#808080',
}

print("Config loaded. Ekman labels:", EKMAN_LABELS)

## 2. Data Loading & Preprocessing

In [ ]:
# ─── 2.1  Load GoEmotions splits ─────────────────────────────────────────────
# Official TSV format (no header): text <tab> comma-separated label ids <tab> comment id.
# The splits are predefined: train fits the models, dev validates, test evaluates.
COLS = ['text', 'labels', 'id']

raw_train = pd.read_csv(DATA_DIR / 'train.tsv', sep='\t', header=None, names=COLS)
raw_val   = pd.read_csv(DATA_DIR / 'dev.tsv',   sep='\t', header=None, names=COLS)
raw_test  = pd.read_csv(DATA_DIR / 'test.tsv',  sep='\t', header=None, names=COLS)

print(f"train: {len(raw_train):,}  |  dev: {len(raw_val):,}  |  test: {len(raw_test):,}")
raw_train.head(3)

In [ ]:
# ─── 2.2  Map 27 GoEmotions emotions → Ekman 6 (+ neutral) ───────────────────
# Label ids index emotions.txt by file order (0-27); neutral is already its last line.
GOEMOTIONS_LABELS = (DATA_DIR / 'emotions.txt').read_text().splitlines()

# Authoritative fine→coarse lookup straight from the dataset's mapping file.
with open(DATA_DIR / 'ekman_mapping.json') as f:
    ekman_groups = json.load(f)
EKMAN_MAP = {fine: coarse for coarse, fines in ekman_groups.items() for fine in fines}
EKMAN_MAP['neutral'] = 'neutral'

def to_ekman(label_ids: str) -> str:
    """Collapse a comma-separated id list to one Ekman label (first listed emotion wins)."""
    fine = [GOEMOTIONS_LABELS[int(i)] for i in str(label_ids).split(',') if i.strip().isdigit()]
    return next((EKMAN_MAP[e] for e in fine if e in EKMAN_MAP), 'neutral')

def to_ekman_frame(raw: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({'text': raw['text'].astype(str)})
    out['ekman_label'] = raw['labels'].apply(to_ekman)
    out['label_id']    = out['ekman_label'].map(LABEL2ID)
    return out

train_df = to_ekman_frame(raw_train)
val_df   = to_ekman_frame(raw_val)
test_df  = to_ekman_frame(raw_test)

print("Train Ekman distribution:")
print(train_df['ekman_label'].value_counts())
train_df.head(5)

In [ ]:
# ─── 2.3  Text preprocessing ─────────────────────────────────────────────────
# Transformers take raw text (the tokeniser handles normalisation); classical
# models get a cleaned version. Negations carry emotion ("not happy"), so they
# stay out of the stoplist and are captured by (1,2)-grams in the TF-IDF step.
_stop = set(stopwords.words('english')) - {'not', 'no', 'never', "don't", "isn't", "wasn't", "wouldn't"}
_lemm = WordNetLemmatizer()

def clean_for_classical(text: str) -> str:
    """Lowercase, lemmatise, drop stopwords (negations kept)."""
    tokens = word_tokenize(text.lower())
    return ' '.join(_lemm.lemmatize(t) for t in tokens if t.isalpha() and t not in _stop)

for d in (train_df, val_df, test_df):
    d['text_clean'] = d['text'].apply(clean_for_classical)

# Combined frame for EDA only — training uses the official splits above.
df_ekman = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

In [ ]:
# ─── 2.4  EDA plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class balance
counts = df_ekman['ekman_label'].value_counts()
colors = [EMOTION_COLORS[l] for l in counts.index]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[0].set_title('Ekman Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Text length distribution
lengths = df_ekman['text'].str.split().str.len()
axes[1].hist(lengths, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(lengths.median(), color='red', linestyle='--', label=f'Median={lengths.median():.0f}')
axes[1].set_title('Token Count Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Words per sample')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Fine-Tuning DistilRoBERTa (Emotion Model)

In [ ]:
# ─── 3.1  Tokeniser & Dataset ─────────────────────────────────────────────────
emotion_tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL_CKPT)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = EmotionDataset(train_df['text'], train_df['label_id'], emotion_tokenizer)
val_dataset   = EmotionDataset(val_df['text'],   val_df['label_id'],   emotion_tokenizer)
test_dataset  = EmotionDataset(test_df['text'],  test_df['label_id'],  emotion_tokenizer)

print(f"Dataset sizes — train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

In [ ]:
# ─── 3.2  Model ──────────────────────────────────────────────────────────────
emotion_model = AutoModelForSequenceClassification.from_pretrained(
    EMOTION_MODEL_CKPT,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
print(f"Parameters: {sum(p.numel() for p in emotion_model.parameters()):,}")

In [ ]:
# ─── 3.3  Metrics ────────────────────────────────────────────────────────────
accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1_macro': f1_metric.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

In [ ]:
# ─── 3.4  Training Arguments ─────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = 'cosine',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    seed                        = SEED,
    report_to                   = 'none',
)

trainer = Trainer(
    model           = emotion_model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

train_result = trainer.train()
print(train_result.metrics)

In [ ]:
# ─── 3.5  Training Curves ────────────────────────────────────────────────────
log_history = pd.DataFrame(trainer.state.log_history).dropna(subset=['eval_loss'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(log_history['epoch'], log_history['eval_loss'],   marker='o', color='tomato')
axes[0].set_title('Validation Loss');    axes[0].set_xlabel('Epoch')
axes[1].plot(log_history['epoch'], log_history['eval_accuracy'], marker='o', color='steelblue')
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch')
axes[2].plot(log_history['epoch'], log_history['eval_f1_macro'], marker='o', color='seagreen')
axes[2].set_title('Validation F1 Macro'); axes[2].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sarcasm Detection (T5, Zero-Shot)

In [ ]:
# ─── 4.1  Load sarcasm model ─────────────────────────────────────────────────
# mrm8488/t5-base-finetuned-sarcasm-twitter is already fine-tuned;
# it takes a sentence and returns "sarcasm" or "not sarcasm".

sarcasm_tokenizer = AutoTokenizer.from_pretrained(SARCASM_MODEL_CKPT)
sarcasm_model     = AutoModelForSeq2SeqLM.from_pretrained(SARCASM_MODEL_CKPT).to(DEVICE)
sarcasm_model.eval()

def detect_sarcasm(text: str) -> dict:
    """
    Returns {'label': 'sarcasm'|'not sarcasm', 'confidence': float}.
    Confidence is derived from the token probability of the first generated token.
    """
    prompt = f"sarcasm: {text}"
    inputs = sarcasm_tokenizer(
        prompt, return_tensors='pt', max_length=128, truncation=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs = sarcasm_model.generate(
            **inputs,
            max_new_tokens=5,
            output_scores=True,
            return_dict_in_generate=True,
        )

    decoded = sarcasm_tokenizer.decode(outputs.sequences[0], skip_special_tokens=True).strip().lower()

    # Approximate confidence: softmax over first token logits
    first_logits = outputs.scores[0][0]          # shape (vocab_size,)
    probs        = torch.softmax(first_logits, dim=-1)
    top_prob     = probs.max().item()

    label = 'sarcasm' if 'sarcasm' in decoded and 'not' not in decoded else 'not sarcasm'
    return {'label': label, 'confidence': round(top_prob, 4)}

# Smoke test
print(detect_sarcasm("Oh great, another Monday. Just what I needed."))
print(detect_sarcasm("I genuinely love spending time with my family."))

## 5. Word Attribution (Integrated Gradients via Captum)

In [ ]:
from captum.attr import LayerIntegratedGradients
import re

emotion_model.eval()
emotion_model.to(DEVICE)

def predict_emotion_with_attribution(text: str, n_steps: int = 50) -> dict:
    """
    Returns:
      emotions      : list of (label, confidence) sorted desc
      tokens        : list of subword tokens
      attributions  : normalised attribution score per token [0, 1]
    """
    enc = emotion_tokenizer(
        text,
        return_tensors='pt',
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    input_ids      = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)

    # Forward pass → get predicted class
    with torch.no_grad():
        logits = emotion_model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs      = torch.softmax(logits, dim=-1).squeeze(0)
    pred_class = probs.argmax().item()

    # Build confidence dict
    emotions = sorted(
        [(ID2LABEL[i], float(probs[i])) for i in range(NUM_LABELS)],
        key=lambda x: x[1], reverse=True
    )

    # ── Integrated Gradients ─────────────────────────────────────────────────
    embed_layer = emotion_model.roberta.embeddings

    def forward_fn(inputs_embeds):
        out = emotion_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
        )
        return out.logits[:, pred_class]

    lig = LayerIntegratedGradients(forward_fn, embed_layer)

    baseline_ids = torch.zeros_like(input_ids)
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    # Aggregate across embedding dim → per-token scalar
    attr = attributions.sum(dim=-1).squeeze(0)          # (seq_len,)
    attr = attr * attention_mask.squeeze(0).float()     # mask padding
    attr = torch.abs(attr)                              # magnitude only

    # Normalise to [0, 1]
    attr_min, attr_max = attr.min(), attr.max()
    if attr_max > attr_min:
        attr = (attr - attr_min) / (attr_max - attr_min)
    attr = attr.cpu().numpy()

    # Decode tokens, strip special tokens and padding
    ids_list = input_ids.squeeze(0).cpu().tolist()
    tokens_raw = emotion_tokenizer.convert_ids_to_tokens(ids_list)

    token_attrs = [
        (t.replace('Ġ', '').replace('▁', ''), float(a))
        for t, a, m in zip(tokens_raw, attr, attention_mask.squeeze(0).cpu().tolist())
        if m == 1 and t not in (emotion_tokenizer.cls_token, emotion_tokenizer.sep_token,
                                 emotion_tokenizer.pad_token, '<s>', '</s>')
    ]

    return {
        'text':         text,
        'emotions':     emotions,
        'top_emotion':  emotions[0][0],
        'confidence':   emotions[0][1],
        'token_attrs':  token_attrs,   # list of (token, score)
    }


def visualise_attribution(result: dict, figsize=(12, 2.5)):
    """Render token-level attribution as a coloured heatmap bar."""
    tokens, scores = zip(*result['token_attrs'])
    scores = np.array(scores)

    fig, ax = plt.subplots(figsize=figsize)
    cmap = plt.cm.YlOrRd
    for i, (tok, sc) in enumerate(zip(tokens, scores)):
        color = cmap(sc)
        ax.text(i, 0, tok, ha='center', va='center', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.85, edgecolor='none'))

    em_color = EMOTION_COLORS.get(result['top_emotion'], 'grey')
    ax.set_xlim(-1, len(tokens))
    ax.set_ylim(-0.6, 0.6)
    ax.axis('off')
    ax.set_title(
        f"Predicted: {result['top_emotion'].upper()}  ({result['confidence']:.1%})  "
        f"— token attribution (darker = higher contribution)",
        color=em_color, fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('attribution_sample.png', dpi=150, bbox_inches='tight')
    plt.show()


# Demo
sample = "I'm absolutely thrilled about this wonderful news!"
result = predict_emotion_with_attribution(sample)
print("Top emotion:", result['top_emotion'], f"({result['confidence']:.1%})")
print("All emotions:", result['emotions'])
visualise_attribution(result)

## 6. Classical ML Baselines

In [ ]:
# ─── 6.1  TF-IDF + Classifiers ───────────────────────────────────────────────
# We use (1,2)-grams to capture negations and common emotion phrases.
# sublinear_tf dampens the effect of very frequent terms (similar to log-TF).

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=60_000,
    sublinear_tf=True,
    min_df=3,
)

X_train_tfidf = tfidf.fit_transform(train_df['text_clean'])
X_val_tfidf   = tfidf.transform(val_df['text_clean'])
X_test_tfidf  = tfidf.transform(test_df['text_clean'])

y_train = train_df['label_id'].values
y_val   = val_df['label_id'].values
y_test  = test_df['label_id'].values

classical_models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, multi_class='multinomial', solver='lbfgs', random_state=SEED
    ),
    'Linear SVM': CalibratedClassifierCV(
        LinearSVC(C=0.5, max_iter=2000, random_state=SEED)
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        n_jobs=-1, random_state=SEED
    ),
    'Naive Bayes': MultinomialNB(alpha=0.5),
}

trained_classical = {}
for name, clf in classical_models.items():
    clf.fit(X_train_tfidf, y_train)
    trained_classical[name] = clf
    val_acc = accuracy_score(y_val, clf.predict(X_val_tfidf))
    print(f"{name:<25} val accuracy: {val_acc:.4f}")

In [ ]:
# ─── 6.2  Classical — token attribution via TF-IDF weights ───────────────────
def classical_word_attribution(text_clean: str, clf, tfidf_vec, top_n: int = 10) -> list:
    """
    Returns top_n (token, score ∈ [0,1]) pairs for Logistic Regression or SVM.
    For tree-based models falls back to TF-IDF feature importance.
    """
    vec     = tfidf_vec.transform([text_clean])
    feat_names = np.array(tfidf_vec.get_feature_names_out())
    tfidf_vals = vec.toarray()[0]

    # Predicted class
    pred = clf.predict(vec)[0]

    # Coef-based (LR, SVM)
    base_clf = clf.estimator if hasattr(clf, 'estimator') else clf
    if hasattr(base_clf, 'coef_'):
        coefs = base_clf.coef_[pred] if base_clf.coef_.ndim > 1 else base_clf.coef_[0]
        scores = tfidf_vals * np.abs(coefs)
    elif hasattr(base_clf, 'feature_importances_'):
        scores = tfidf_vals * base_clf.feature_importances_
    else:
        scores = tfidf_vals

    non_zero = np.nonzero(scores)[0]
    if len(non_zero) == 0:
        return []

    top_idx   = non_zero[np.argsort(scores[non_zero])[::-1][:top_n]]
    top_scores = scores[top_idx]
    # Normalise
    if top_scores.max() > 0:
        top_scores = top_scores / top_scores.max()

    return list(zip(feat_names[top_idx].tolist(), top_scores.tolist()))

## 7. Evaluation & Comparison

In [ ]:
# ─── 7.1  DistilRoBERTa test-set inference ───────────────────────────────────
test_outputs   = trainer.predict(test_dataset)
roberta_preds  = np.argmax(test_outputs.predictions, axis=-1)
roberta_probs  = torch.softmax(torch.tensor(test_outputs.predictions), dim=-1).numpy()

roberta_acc    = accuracy_score(y_test, roberta_preds)
roberta_f1     = f1_score(y_test, roberta_preds, average='macro')
print(f"DistilRoBERTa — Accuracy: {roberta_acc:.4f}  |  Macro F1: {roberta_f1:.4f}")

In [ ]:
# ─── 7.2  Summary comparison table ──────────────────────────────────────────
rows = []
for name, clf in trained_classical.items():
    preds = clf.predict(X_test_tfidf)
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Macro F1': f1_score(y_test, preds, average='macro'),
        'Weighted F1': f1_score(y_test, preds, average='weighted'),
    })

rows.append({
    'Model': 'DistilRoBERTa (fine-tuned)',
    'Accuracy': roberta_acc,
    'Macro F1': roberta_f1,
    'Weighted F1': f1_score(y_test, roberta_preds, average='weighted'),
})

comparison_df = pd.DataFrame(rows).sort_values('Macro F1', ascending=False)
comparison_df = comparison_df.set_index('Model').round(4)
print(comparison_df.to_string())

# Bar chart
ax = comparison_df[['Accuracy', 'Macro F1', 'Weighted F1']].plot(
    kind='bar', figsize=(11, 5), color=['#4C72B0','#DD8452','#55A868'],
    edgecolor='white', width=0.65
)
ax.set_title('Model Comparison on Test Set', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 7.3  Confusion matrices ─────────────────────────────────────────────────
all_models_for_cm = {
    **{k: v.predict(X_test_tfidf) for k, v in trained_classical.items()},
    'DistilRoBERTa': roberta_preds,
}

n_models = len(all_models_for_cm)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))

for ax, (name, preds) in zip(axes, all_models_for_cm.items()):
    cm = confusion_matrix(y_test, preds, normalize='true')
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=EKMAN_LABELS, yticklabels=EKMAN_LABELS,
        ax=ax, linewidths=0.5, cbar=False,
        annot_kws={'size': 7}
    )
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)

plt.suptitle('Normalised Confusion Matrices', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 7.4  Per-class classification report (DistilRoBERTa) ────────────────────
print("DistilRoBERTa — Per-class Report")
print(classification_report(
    y_test, roberta_preds,
    target_names=EKMAN_LABELS,
    digits=4
))

## 8. Save Artefacts for Dashboard

In [ ]:
import pickle

# Save fine-tuned emotion model + tokeniser
emotion_model.save_pretrained('./emotion_model_final')
emotion_tokenizer.save_pretrained('./emotion_model_final')

# Save classical models + vectoriser
with open('classical_models.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf, 'models': trained_classical}, f)

# Save config
with open('config.json', 'w') as f:
    json.dump({
        'ekman_labels': EKMAN_LABELS,
        'label2id': LABEL2ID,
        'id2label': ID2LABEL,
        'emotion_colors': EMOTION_COLORS,
        'sarcasm_model': SARCASM_MODEL_CKPT,
    }, f, indent=2)

print("All artefacts saved.")

## 9. Streamlit Dashboard

Run with: `streamlit run emotion_dashboard.py`

In [ ]:
dashboard_code = '''
# emotion_dashboard.py
# Run: streamlit run emotion_dashboard.py

import json
import pickle
import re
import numpy as np
import pandas as pd
import torch
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
)
from captum.attr import LayerIntegratedGradients
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

# ─── Page config ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Emotion & Sarcasm Analyzer",
    page_icon="🎭",
    layout="wide",
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── Load config ─────────────────────────────────────────────────────────────
@st.cache_resource
def load_config():
    with open("config.json") as f:
        return json.load(f)

cfg = load_config()
EKMAN_LABELS   = cfg["ekman_labels"]
ID2LABEL       = {int(k): v for k, v in cfg["id2label"].items()}
EMOTION_COLORS = cfg["emotion_colors"]
MAX_LEN        = 128

# ─── Load models ─────────────────────────────────────────────────────────────
@st.cache_resource
def load_emotion_model():
    tok   = AutoTokenizer.from_pretrained("./emotion_model_final")
    model = AutoModelForSequenceClassification.from_pretrained("./emotion_model_final").to(DEVICE)
    model.eval()
    return tok, model

@st.cache_resource
def load_sarcasm_model():
    tok   = AutoTokenizer.from_pretrained(cfg["sarcasm_model"])
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg["sarcasm_model"]).to(DEVICE)
    model.eval()
    return tok, model

@st.cache_resource
def load_classical():
    with open("classical_models.pkl", "rb") as f:
        return pickle.load(f)

emotion_tok, emotion_mdl  = load_emotion_model()
sarcasm_tok, sarcasm_mdl  = load_sarcasm_model()
classical_bundle          = load_classical()
tfidf                     = classical_bundle["tfidf"]
classical_models          = classical_bundle["models"]

_stop = set(stopwords.words("english")) - {"not","no","never"}
_lemm = WordNetLemmatizer()

def clean_text(text):
    tokens = word_tokenize(text.lower())
    return " ".join(_lemm.lemmatize(t) for t in tokens if t.isalpha() and t not in _stop)

# ─── Inference helpers ───────────────────────────────────────────────────────
def predict_emotion(text):
    enc = emotion_tok(
        text, return_tensors="pt", max_length=MAX_LEN,
        truncation=True, padding="max_length"
    )
    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits = emotion_mdl(input_ids=input_ids, attention_mask=attention_mask).logits
    probs      = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    pred_class = int(np.argmax(probs))

    # Integrated Gradients attribution
    embed_layer = emotion_mdl.roberta.embeddings

    def fwd(inputs_embeds):
        return emotion_mdl(
            inputs_embeds=inputs_embeds, attention_mask=attention_mask
        ).logits[:, pred_class]

    lig = LayerIntegratedGradients(fwd, embed_layer)
    baseline = torch.zeros_like(input_ids)
    attr, _ = lig.attribute(input_ids, baselines=baseline, n_steps=30,
                             return_convergence_delta=True)
    attr = attr.sum(dim=-1).squeeze(0) * attention_mask.squeeze(0).float()
    attr = torch.abs(attr).cpu().numpy()
    if attr.max() > attr.min():
        attr = (attr - attr.min()) / (attr.max() - attr.min())

    ids_list   = input_ids.squeeze(0).cpu().tolist()
    tokens_raw = emotion_tok.convert_ids_to_tokens(ids_list)
    mask_list  = attention_mask.squeeze(0).cpu().tolist()
    special    = {emotion_tok.cls_token, emotion_tok.sep_token,
                  emotion_tok.pad_token, "<s>", "</s>"}
    token_attrs = [
        (t.replace("Ġ","").replace("▁",""), float(a))
        for t, a, m in zip(tokens_raw, attr, mask_list)
        if m == 1 and t not in special and t.strip()
    ]

    emotions = [(ID2LABEL[i], float(probs[i])) for i in range(len(probs))]
    emotions.sort(key=lambda x: x[1], reverse=True)
    return emotions, token_attrs


def detect_sarcasm(text):
    inputs = sarcasm_tok(
        f"sarcasm: {text}", return_tensors="pt", max_length=128, truncation=True
    ).to(DEVICE)
    with torch.no_grad():
        out = sarcasm_mdl.generate(
            **inputs, max_new_tokens=5,
            output_scores=True, return_dict_in_generate=True,
        )
    decoded  = sarcasm_tok.decode(out.sequences[0], skip_special_tokens=True).lower()
    top_prob = torch.softmax(out.scores[0][0], dim=-1).max().item()
    label    = "🙃 Sarcasm" if "sarcasm" in decoded and "not" not in decoded else "✅ Not Sarcasm"
    return label, round(top_prob, 4)


def classical_predict(text):
    cleaned = clean_text(text)
    vec     = tfidf.transform([cleaned])
    results = {}
    feat_names = np.array(tfidf.get_feature_names_out())
    tfidf_vals = vec.toarray()[0]

    for name, clf in classical_models.items():
        pred        = clf.predict(vec)[0]
        probs_arr   = clf.predict_proba(vec)[0]
        emotions    = list(zip([ID2LABEL[i] for i in range(len(probs_arr))], probs_arr.tolist()))
        emotions.sort(key=lambda x: x[1], reverse=True)

        # Attribution
        base_clf = clf.estimator if hasattr(clf, "estimator") else clf
        if hasattr(base_clf, "coef_"):
            coefs  = base_clf.coef_[pred] if base_clf.coef_.ndim > 1 else base_clf.coef_[0]
            scores = tfidf_vals * np.abs(coefs)
        elif hasattr(base_clf, "feature_importances_"):
            scores = tfidf_vals * base_clf.feature_importances_
        else:
            scores = tfidf_vals

        non_zero = np.nonzero(scores)[0]
        if len(non_zero) > 0:
            top_idx = non_zero[np.argsort(scores[non_zero])[::-1][:12]]
            top_s   = scores[top_idx]
            if top_s.max() > 0:
                top_s = top_s / top_s.max()
            token_attrs = list(zip(feat_names[top_idx].tolist(), top_s.tolist()))
        else:
            token_attrs = []

        results[name] = {"emotions": emotions, "token_attrs": token_attrs}
    return results


# ─── Plotly helpers ──────────────────────────────────────────────────────────
def confidence_bar(emotions, title=""):
    labels = [e[0] for e in emotions]
    scores = [e[1] for e in emotions]
    colors = [EMOTION_COLORS.get(l, "#888") for l in labels]
    fig = go.Figure(go.Bar(
        x=scores, y=labels, orientation="h",
        marker_color=colors,
        text=[f"{s:.1%}" for s in scores],
        textposition="outside",
    ))
    fig.update_layout(
        title=title, xaxis_range=[0, 1.15],
        xaxis_title="Confidence",
        height=300, margin=dict(l=10, r=20, t=40, b=10),
        plot_bgcolor="rgba(0,0,0,0)",
    )
    return fig


def attribution_chart(token_attrs, title="Token Attribution"):
    if not token_attrs:
        return None
    tokens, scores = zip(*token_attrs)
    fig = go.Figure(go.Bar(
        x=list(tokens), y=list(scores),
        marker=dict(
            color=list(scores),
            colorscale="YlOrRd",
            cmin=0, cmax=1,
            colorbar=dict(title="Attribution", thickness=12),
        ),
    ))
    fig.update_layout(
        title=title,
        yaxis_title="Attribution Score",
        yaxis_range=[0, 1.1],
        height=280, margin=dict(l=10, r=10, t=40, b=50),
        plot_bgcolor="rgba(0,0,0,0)",
    )
    return fig


# ─── UI ──────────────────────────────────────────────────────────────────────
st.title("🎭 Emotion & Sarcasm Analyzer")
st.markdown(
    "Enter any English text to see **Ekman emotion predictions** with confidence scores "
    "and **token-level attribution**, plus **sarcasm detection**."
)

with st.form(key="analysis_form"):
    user_text = st.text_area(
        "Enter text",
        placeholder="e.g. Oh sure, another meeting that could have been an email!",
        height=100,
    )
    submitted = st.form_submit_button("Analyze", type="primary")

if submitted and user_text.strip():
    with st.spinner("Running models..."):
        emotions, token_attrs = predict_emotion(user_text)
        sarcasm_label, sarcasm_conf = detect_sarcasm(user_text)
        classical_results = classical_predict(user_text)

    # ── Top result banner ────────────────────────────────────────────────────
    top_em   = emotions[0][0]
    top_conf = emotions[0][1]
    em_color = EMOTION_COLORS.get(top_em, "#888")

    st.markdown(f"""
    <div style="background:{em_color}22;border-left:5px solid {em_color};
                padding:12px 18px;border-radius:6px;margin-bottom:12px;">
        <h3 style="margin:0;color:{em_color}">
            {top_em.upper()} &nbsp; <span style="font-size:0.8em;color:#555">{top_conf:.1%} confidence</span>
        </h3>
        <p style="margin:4px 0 0 0;color:#555">Sarcasm: <b>{sarcasm_label}</b> &nbsp;({sarcasm_conf:.1%} token prob)</p>
    </div>
    """, unsafe_allow_html=True)

    # ── DistilRoBERTa section ────────────────────────────────────────────────
    st.subheader("🤖 DistilRoBERTa (fine-tuned)")
    c1, c2 = st.columns([1, 1])
    with c1:
        st.plotly_chart(confidence_bar(emotions, "Emotion Confidence"),
                        use_container_width=True)
    with c2:
        fig_attr = attribution_chart(token_attrs, "Token Attribution (Integrated Gradients)")
        if fig_attr:
            st.plotly_chart(fig_attr, use_container_width=True)

    # ── Classical section ────────────────────────────────────────────────────
    st.subheader("📊 Classical ML Baselines")
    cols = st.columns(len(classical_results))

    for col, (name, res) in zip(cols, classical_results.items()):
        with col:
            st.markdown(f"**{name}**")
            top = res["emotions"][0]
            st.metric(label=top[0], value=f"{top[1]:.1%}")
            st.plotly_chart(
                confidence_bar(res["emotions"][:5], ""),
                use_container_width=True,
            )
            fig_ca = attribution_chart(res["token_attrs"][:8], "Top contributing n-grams")
            if fig_ca:
                st.plotly_chart(fig_ca, use_container_width=True)

    # ── Raw data expander ────────────────────────────────────────────────────
    with st.expander("Raw prediction data"):
        st.json({
            "distilroberta": {
                "emotions":    {e: round(c, 4) for e, c in emotions},
                "token_attrs": {t: round(s, 4) for t, s in token_attrs},
            },
            "sarcasm": {"label": sarcasm_label, "confidence": sarcasm_conf},
            "classical": {
                name: {"top": res["emotions"][0][0],
                        "confidence": round(res["emotions"][0][1], 4)}
                for name, res in classical_results.items()
            },
        })

elif submitted:
    st.warning("Please enter some text first.")
'''

with open('emotion_dashboard.py', 'w') as f:
    f.write(dashboard_code.strip())

print("Dashboard saved → emotion_dashboard.py")
print("Run with: streamlit run emotion_dashboard.py")

## 10. Quick End-to-End Demo

In [ ]:
DEMO_TEXTS = [
    "Oh great, another meeting that could have been an email. I'm thrilled.",
    "I just got the promotion I worked so hard for. Today is amazing!",
    "I can't believe they cancelled the show. I'm absolutely devastated.",
    "This is fine. Everything is completely fine. Nothing is on fire.",
    "Why would anyone do something so cruel? This makes me furious.",
]

print(f"{'TEXT':<55} {'TOP EMOTION':<12} {'CONF':>6}  {'SARCASM':<16}")
print("-" * 95)
for txt in DEMO_TEXTS:
    emotions, _ = predict_emotion(txt)
    sarc_lbl, sarc_conf = detect_sarcasm(txt)
    top_em   = emotions[0][0]
    top_conf = emotions[0][1]
    print(f"{txt[:54]:<55} {top_em:<12} {top_conf:>6.1%}  {sarc_lbl}")